In [1]:
import time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, matthews_corrcoef, recall_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
# !pip install imbalanced-learn

P = Path("processed")
X_train, y_train = np.load(P/"X_train.npy"), np.load(P/"y_train.npy")
print(X_train.shape, "attack share:", y_train.mean().round(3))

(140311, 68) attack share: 0.864


In [2]:
N_SUB, N_TREES = 40_000, 40
X_s, _, y_s, _ = train_test_split(
    X_train, y_train, train_size=N_SUB, stratify=y_train, random_state=42)

In [3]:
def normal_recall(yt, yp):            # recall of class 0 (normal)
    return recall_score(yt, yp, pos_label=0)

methods = {
    "none":             [],
    "class_weight":     [],                       # handled in the model below
    "random_over":      [("s", RandomOverSampler(random_state=42))],
    "random_under":     [("s", RandomUnderSampler(random_state=42))],
    "smote":            [("s", SMOTE(random_state=42))],
    "borderline_smote": [("s", BorderlineSMOTE(random_state=42))],
    "under_then_smote": [("u", RandomUnderSampler(sampling_strategy=0.5, random_state=42)),
                         ("s", SMOTE(random_state=42))],
    "smote_tomek":      [("s", SMOTETomek(random_state=42))],
}

scoring = {
    "recall_attack": "recall",
    "normal_recall": make_scorer(normal_recall),
    "precision":     "precision",
    "f1":            "f1",
    "macro_f1":      "f1_macro",
    "pr_auc":        "average_precision",
    "mcc":           make_scorer(matthews_corrcoef),
}

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, steps in methods.items():
    model = RandomForestClassifier(
        n_estimators=N_TREES, n_jobs=-1, random_state=42,
        class_weight="balanced" if name == "class_weight" else None)
    pipe = Pipeline(steps + [("clf", model)])     # resampling happens inside each fold only
    res = cross_validate(pipe, X_s, y_s, cv=cv, scoring=scoring)
    row = {"method": name, "fit_s": res["fit_time"].mean()}
    for k in scoring:
        row[k] = res["test_" + k].mean()
        row[k + "_sd"] = res["test_" + k].std()
    rows.append(row)
    print(name, "done")

results = pd.DataFrame(rows).sort_values("mcc", ascending=False)
results[["method","recall_attack","normal_recall","precision","macro_f1",
         "pr_auc","mcc","mcc_sd","fit_s"]].round(4)

none done
class_weight done
random_over done
random_under done
smote done
borderline_smote done
under_then_smote done
smote_tomek done


,method,recall_attack,normal_recall,precision,macro_f1,pr_auc,mcc,mcc_sd,fit_s
0,none,0.9976,0.9522,0.9925,0.9816,0.9965,0.9633,0.0029,0.5749
2,random_over,0.9948,0.9554,0.9930,0.9773,0.9966,0.9547,0.0034,1.4492
1,class_weight,0.9933,0.9572,0.9933,0.9754,0.9972,0.9507,0.0024,0.5076
7,smote_tomek,0.9928,0.9598,0.9937,0.9752,0.9973,0.9503,0.0032,9.6106
4,smote,0.9930,0.9583,0.9935,0.9750,0.9975,0.9501,0.0013,1.3095
5,borderline_smote,0.9883,0.9554,0.9930,0.9660,0.9970,0.9321,0.0026,2.0344
6,under_then_smote,0.9859,0.9622,0.9940,0.9639,0.9976,0.9283,0.0022,0.6097
3,random_under,0.9774,0.9639,0.9942,0.9502,0.9984,0.9022,0.0127,0.2121


In [5]:
results.round(4).to_csv(P/"balancing_comparison.csv", index=False)

In [6]:
X_val, y_val = np.load(P/"X_val.npy"), np.load(P/"y_val.npy")
from sklearn.metrics import classification_report, confusion_matrix

for name in ["none", "class_weight"]:
    clf = RandomForestClassifier(
        n_estimators=200, n_jobs=-1, random_state=42,
        class_weight="balanced" if name == "class_weight" else None)
    clf.fit(X_train, y_train)                       # full training set
    pred = clf.predict(X_val)
    print("=====", name)
    print(confusion_matrix(y_val, pred))
    print(classification_report(y_val, pred, target_names=["normal","attack"], digits=4))

===== none
[[ 3926   151]
 [   33 25957]]
              precision    recall  f1-score   support

      normal     0.9917    0.9630    0.9771      4077
      attack     0.9942    0.9987    0.9965     25990

    accuracy                         0.9939     30067
   macro avg     0.9929    0.9808    0.9868     30067
weighted avg     0.9939    0.9939    0.9938     30067

===== class_weight
[[ 3943   134]
 [  192 25798]]
              precision    recall  f1-score   support

      normal     0.9536    0.9671    0.9603      4077
      attack     0.9948    0.9926    0.9937     25990

    accuracy                         0.9892     30067
   macro avg     0.9742    0.9799    0.9770     30067
weighted avg     0.9892    0.9892    0.9892     30067

